# BERT の文脈化ベクトルで単語の使われ方の変化を調べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_lexical_semantics/semantic_change_bert.ipynb)

NLP クックブック 18-14 ／ 語の意味と意味の変化

1 つ 1 つの用例を文脈化ベクトルにして、時代ごとの用法の広がりや割合の変化を調べたい。

In [1]:
!pip install -q datasets fugashi unidic-lite scikit-learn

In [2]:
import random, re
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer

ds = load_dataset("globis-university/aozorabunko-clean", split="train")
random.seed(0)
TARGETS = ["原子", "同志", "活動", "言葉", "必要"]

def era(meta):
    year = (meta.get("生年月日") or "")[:4]
    if meta["文字遣い種別"] != "新字新仮名" or not year.isdigit():
        return None
    return "旧" if 1850 <= int(year) < 1880 else "新" if 1895 <= int(year) < 1930 else None

# 対象語を含む文 (用例) を、時代ごとに最大 150 件ずつ集める
usages = {(t, e): [] for t in TARGETS for e in ["旧", "新"]}
order = list(range(len(ds))); random.shuffle(order)
for i in order:
    e = era(ds[i]["meta"])
    if e is None:
        continue
    per_work = {t: 0 for t in TARGETS}
    for s in re.split(r"(?<=[。！？])|\n", ds[i]["text"][:30000]):
        for t in TARGETS:
            if t in s and 10 <= len(s) <= 120 and len(usages[t, e]) < 150 and per_work[t] < 3:     # 1 作品からは 3 件まで
                usages[t, e].append(s.strip()); per_work[t] += 1
    if all(len(u) >= 150 for u in usages.values()):
        break
print({k: len(v) for k, v in usages.items()})

{('原子', '旧'): 43, ('原子', '新'): 150, ('同志', '旧'): 116, ('同志', '新'): 150, ('活動', '旧'): 150, ('活動', '新'): 150, ('言葉', '旧'): 150, ('言葉', '新'): 150, ('必要', '旧'): 150, ('必要', '新'): 150}


In [3]:
name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval()

@torch.no_grad()
def usage_vectors(word, sentences, batch_size=32):
    """文の中の対象語の位置にあたるトークンのベクトル (最終層) を取り出す"""
    word_ids = tokenizer(word, add_special_tokens=False)["input_ids"]
    vectors = []
    for i in range(0, len(sentences), batch_size):
        enc = tokenizer(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128)
        hidden = model(**enc).last_hidden_state
        for ids, h in zip(enc["input_ids"].tolist(), hidden):
            for j in range(len(ids) - len(word_ids) + 1):
                if ids[j:j + len(word_ids)] == word_ids:
                    vectors.append(h[j:j + len(word_ids)].mean(0).numpy()); break
    return np.stack(vectors)

vectors = {k: usage_vectors(k[0], v) for k, v in usages.items()}
print({k: v.shape[0] for k, v in vectors.items()})

{('原子', '旧'): 41, ('原子', '新'): 148, ('同志', '旧'): 116, ('同志', '新'): 150, ('活動', '旧'): 150, ('活動', '新'): 150, ('言葉', '旧'): 145, ('言葉', '新'): 150, ('必要', '旧'): 150, ('必要', '新'): 150}


In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances

def prt(a, b):
    """時代ごとの平均ベクトル (プロトタイプ) どうしの距離"""
    return float(cosine_distances(a.mean(0, keepdims=True), b.mean(0, keepdims=True))[0, 0])

def apd(a, b):
    """時代をまたぐ用例の組の、平均距離 (average pairwise distance)"""
    return float(cosine_distances(a, b).mean())

def jsd(a, b, k=4):
    """全用例をクラスタリングして、各クラスタ (≒ 用法) の割合の分布を時代間で比べる (Jensen-Shannon 距離)"""
    labels = KMeans(k, n_init=10, random_state=0).fit_predict(np.vstack([a, b]))
    p = np.bincount(labels[:len(a)], minlength=k) / len(a); q = np.bincount(labels[len(a):], minlength=k) / len(b)
    m = (p + q) / 2
    kl = lambda x, y: float(np.sum(x[x > 0] * np.log2(x[x > 0] / y[x > 0])))
    return (kl(p, m) + kl(q, m)) / 2, labels, p, q

for t in TARGETS:
    a, b = vectors[t, "旧"], vectors[t, "新"]
    print(f"{t}: PRT {prt(a, b):.3f} / APD {apd(a, b):.3f} / 時代内の平均距離 旧 {apd(a, a):.3f} 新 {apd(b, b):.3f} / JSD {jsd(a, b)[0]:.3f}")

原子: PRT 0.050 / APD 0.236 / 時代内の平均距離 旧 0.183 新 0.208 / JSD 0.321
同志: PRT 0.009 / APD 0.255 / 時代内の平均距離 旧 0.258 新 0.239 / JSD 0.042
活動: PRT 0.009 / APD 0.233 / 時代内の平均距離 旧 0.228 新 0.224 / JSD 0.057
言葉: PRT 0.005 / APD 0.220 / 時代内の平均距離 旧 0.220 新 0.213 / JSD 0.022
必要: PRT 0.004 / APD 0.270 / 時代内の平均距離 旧 0.262 新 0.272 / JSD 0.003


In [5]:
# クラスタごとに用例を見て、どの用法が増減したかを確かめる
def show_clusters(word, k=4, n=2):
    a, b = vectors[word, "旧"], vectors[word, "新"]
    _, labels, p, q = jsd(a, b, k)
    sents = usages[word, "旧"][:len(a)] + usages[word, "新"][:len(b)]
    for c in np.argsort(q - p):
        print(f"[クラスタ {c}] 旧 {p[c]:.0%} → 新 {q[c]:.0%}")
        for s in [s for s, l in zip(sents, labels) if l == c][:n]:
            print("    ", s[:70])

show_clusters("同志")
show_clusters("原子")

[クラスタ 0] 旧 32% → 新 20%
     老婦人の多く集る諸種の会合はあっても、それは凡て物見遊山の変形で、老婦人同志の奢侈と自慢の競進場たるに過ぎない。
     従兄妹同志の間ではあるが日頃正太のことを「兄さん、兄さん」と呼んでいた。
[クラスタ 3] 旧 7% → 新 1%
     私は、大正三年九月にヨーロッパから帰ると、当時、京都に創始せられることになった同志社大学に、教授として就職することになった。
     同志社大学では、満三年のあいだ、国際法と外交史の講座を担当していた。
[クラスタ 2] 旧 22% → 新 24%
     若いもの同志の話は木曾少女の美しいことに落ちて行った。
     けれども、それは何、少いもの同志だから、萌黄縅の鎧はなくても、夜一夜、戸外を歩行いていたって、それで事は済みました。
[クラスタ 1] 旧 39% → 新 55%
     で、私は今日の会も堺氏の出獄を期して、われら同志友人がここに一日の園遊会を開いたという風に思い」たいと語った。
     勿論知らぬ同志であるから多少気まずい点はあろうと考えていたが、斯く迄に排他的であろうとは全く予想しなかった。
[クラスタ 1] 旧 76% → 新 16%
     そして原子の衝突についてなんとか自分流儀の解釈をしてみようと思ってだいぶ骨折ってみたが、どうもうまくできない」と言っている。
     原子素量の存在、その結合による物質の構成機巧、物質総量の不滅、原子の運動衝突と物性の関係、そういうようなものが予想されているばかりでなく、見
[クラスタ 3] 旧 15% → 新 20%
     アンドラデによると、ニウトンの原子に関する説明（２）を読めば、彼がルクレチウスを知らなかったと想像する事はできないということである。
     その不思議を昔われらの先祖が化け物へ帰納したのを、今の科学者は分子原子電子へ持って行くだけの事である。
[クラスタ 0] 旧 5% → 新 30%
     またこれら分子がまた原子から成立している事も疑いない事で、分子中における原子結合の状況についても各方面から推定を下す手掛りが出来ている。
     脳髄の中にある原子の数はたぶん十の二十何乗という莫大な数であろう。
[クラスタ 2] 旧 5% → 新 34%
    

- 単語ベクトル (word2vec) は 1 語に 1 つのベクトルしかなく、複数の語義が混ざる。文脈化ベクトルは用例ごとに別のベクトルになるので、「どの用法が増えたか」まで調べられる。
- PRT は平均どうしの距離、APD は用例の組の平均距離、JSD は用法（クラスタ）の割合の変化。同じ語でも指標によって順位が変わるので、複数を並べて見る。
- 「同志」は、単語ベクトルでは大きく変化して見えたが、ここではほとんど差が出ない。用例を読むと、どちらの時代も「若いもの同志」のような「同士」の意味が大半で、政治的な用法は一部にすぎない。少数の新しい用法は、平均や距離の指標には表れにくい。
- 学習済みの BERT は現代語のテキストで学習されている。古い時代の文章や表記には弱く、文体の違いが「意味の変化」として表れてしまうことがある。
- 用例の数が少ないと、どの指標も不安定になる。時代ごとに 100 件以上を目安にし、1 つの作品に偏らないように集める。
- クラスタは語義と一致するとは限らない。文体や構文の違いで分かれることも多いので、必ず用例を読んで確かめる。